# 01 — Data Ingestion & Preprocessing (STLF Ecuador SNI)

In [1]:
!pip install -q holidays

In [2]:
# Project root: Google Drive on Colab, current directory when running locally
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Project_IA")
except ImportError:
    BASE_DIR = Path.cwd()

BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {BASE_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Project_IA


In [3]:
# Imports & configuration
import holidays
import numpy as np
import pandas as pd
import requests

RAW_DIR = BASE_DIR / "datos_demanda"
DATA_DIR = BASE_DIR / "data"
for d in (RAW_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

MONTHLY_PATH = RAW_DIR / "demanda_mensual_SNI_2020_2025.csv"
RECON_PATH = RAW_DIR / "demanda_horaria_reconstruida.csv"
OUTPUT_PATH = DATA_DIR / "dataset_stlf_ecuador.csv"
WEATHER_CACHE = DATA_DIR / "weather_open_meteo_cache.csv"

TZ = "America/Guayaquil"
TARGET = "demand_mw"  # English name used in the output dataset
# Column names of the Spanish source CSVs (kept as-is on input)
RAW_TIME_COL = "fecha_hora"
RAW_TARGET_COL = "demanda_mw"
HORIZON = 24  # day-ahead: the 24 hours of day D are forecast at 23:00 of day D-1
N_LAGS = 72   # demand history known at the forecast origin (PACF analysis, CENACE)
# lag_1h ... lag_95h: the model for lead time h uses lag_h ... lag_(h+71), i.e. the 72 hours known at the origin
LAGS = range(1, HORIZON + N_LAGS)
SEED = 42

# Main SNI load centers (paper, Section 7)
CITIES = {
    "guayaquil": (-2.1969, -79.8862),
    "quito": (-0.1807, -78.4678),
    "cuenca": (-2.9001, -79.0059),
}

In [4]:
# Ecuador official holidays (holidays library) honoring the transfer law:
# when a holiday is moved ("observed"), only the transferred date is a day off.
OBSERVED_SUFFIX = r"\s*\((?:observed|observado)\)$"


def ecuador_holiday_dates(years, extra=None):
    cal = holidays.country_holidays("EC", years=list(years), language="en_US")
    table = pd.DataFrame(sorted(cal.items()), columns=["date", "holiday"])
    table["date"] = pd.to_datetime(table["date"])
    table["is_observed"] = table["holiday"].str.contains(OBSERVED_SUFFIX, regex=True)
    table["base_name"] = table["holiday"].str.replace(OBSERVED_SUFFIX, "", regex=True)

    moved = set(zip(table.loc[table["is_observed"], "date"].dt.year,
                    table.loc[table["is_observed"], "base_name"]))
    is_moved_original = [
        (d.year, n) in moved and not o
        for d, n, o in zip(table["date"], table["base_name"], table["is_observed"])
    ]
    table = table.loc[~np.array(is_moved_original), ["date", "holiday"]]

    if extra:  # executive-decree bridge days: {"YYYY-MM-DD": "name"}
        table = pd.concat([table, pd.DataFrame({"date": pd.to_datetime(list(extra)),
                                                "holiday": list(extra.values())})])
    return table.drop_duplicates("date").sort_values("date").reset_index(drop=True)


EXTRA_HOLIDAYS = {}

In [5]:
# 0. Hourly reconstruction: monthly energy (GWh) -> hourly demand (MW)
#    Typical SNI daily profile (24 h) x day-of-week modulation (working day vs weekend/holiday),
#    rescaled so every month's hourly energy matches the official CENACE monthly total.
RECON_START, RECON_END = "2020-12-01 00:00", "2025-12-31 23:00"  # Dec-2020 = lag warm-up (up to 95 h)
NOISE_STD = 0.02   # relative AR(1) noise (0 = purely deterministic profile)
NOISE_PHI = 0.90

# Per-unit daily shapes: valley ~03-04 h, evening peak ~19-20 h
PROFILE_WORKDAY = np.array([0.84, 0.80, 0.78, 0.77, 0.78, 0.82, 0.88, 0.93, 0.98, 1.02, 1.05, 1.07,
                            1.06, 1.05, 1.06, 1.06, 1.05, 1.05, 1.12, 1.21, 1.19, 1.12, 1.02, 0.92])
PROFILE_NON_WORKING = np.array([0.88, 0.84, 0.82, 0.80, 0.80, 0.81, 0.84, 0.87, 0.92, 0.97, 1.00, 1.02,
                                1.03, 1.02, 1.01, 1.00, 1.00, 1.02, 1.12, 1.22, 1.21, 1.15, 1.06, 0.96])
DAY_OF_WEEK_FACTOR = np.array([1.00, 1.01, 1.01, 1.01, 1.00, 0.93, 0.86])  # Mon ... Sun
HOLIDAY_FACTOR = 0.85

monthly = pd.read_csv(MONTHLY_PATH, parse_dates=["fecha"])
monthly_mwh = monthly.set_index(monthly["fecha"].dt.to_period("M"))["demanda_gwh"] * 1_000

idx = pd.date_range(RECON_START, RECON_END, freq="h")
recon_holidays = ecuador_holiday_dates(range(idx.year.min(), idx.year.max() + 1), EXTRA_HOLIDAYS)

hours = idx.hour.to_numpy()
dow = idx.dayofweek.to_numpy()
is_hol = idx.normalize().isin(recon_holidays["date"])
non_working = (dow >= 5) | is_hol

weights = np.where(non_working, PROFILE_NON_WORKING[hours], PROFILE_WORKDAY[hours])
weights = weights * np.where(is_hol, HOLIDAY_FACTOR, DAY_OF_WEEK_FACTOR[dow])

rng = np.random.default_rng(SEED)
innov = rng.normal(0.0, NOISE_STD * np.sqrt(1 - NOISE_PHI**2), len(idx))
noise = np.zeros(len(idx))
for t in range(1, len(idx)):
    noise[t] = NOISE_PHI * noise[t - 1] + innov[t]
weights = pd.Series(weights * (1 + noise), index=idx)

period = idx.to_period("M")
month_mwh = monthly_mwh.reindex(period).to_numpy()
if np.isnan(month_mwh).any():
    raise ValueError(f"Missing monthly totals for: {sorted(set(period[np.isnan(month_mwh)].astype(str)))}")

hourly_mw = weights * month_mwh / weights.groupby(period).transform("sum").to_numpy()

# Saved in the same (Spanish) format as the raw hourly CSVs it stands in for
recon = pd.DataFrame({RAW_TIME_COL: idx, RAW_TARGET_COL: hourly_mw.round(2).to_numpy()})
recon.to_csv(RECON_PATH, index=False)

check = (recon.groupby(period)[RAW_TARGET_COL].sum() / 1_000).to_frame("reconstructed_gwh")
check["official_gwh"] = monthly_mwh.reindex(check.index) / 1_000
check["abs_diff_gwh"] = (check["reconstructed_gwh"] - check["official_gwh"]).abs()
print(f"Saved -> {RECON_PATH} | {len(recon):,} h | max monthly energy diff: {check['abs_diff_gwh'].max():.4f} GWh")
recon[RAW_TARGET_COL].describe()

Saved -> /content/drive/MyDrive/Project_IA/datos_demanda/demanda_horaria_reconstruida.csv | 44,568 h | max monthly energy diff: 0.0001 GWh


,demanda_mw
count,44568.000000
mean,3376.061313
std,595.810927
min,1914.740000
25%,2927.070000
50%,3349.340000
75%,3825.620000
max,5202.250000


In [6]:
# 1. Load and consolidate hourly-demand CSVs (Spanish source columns -> English names)
REQUIRED_COLS = {RAW_TIME_COL, RAW_TARGET_COL}

frames = []
for csv_path in sorted(RAW_DIR.glob("*.csv")):
    raw = pd.read_csv(csv_path)
    missing = REQUIRED_COLS - set(raw.columns)
    if missing:
        print(f"[skip] {csv_path.name}: missing columns {sorted(missing)}")
        continue
    frames.append(raw[[RAW_TIME_COL, RAW_TARGET_COL]])

if not frames:
    raise FileNotFoundError(
        f"No CSV in '{RAW_DIR}' has the columns {sorted(REQUIRED_COLS)} (hourly demand)."
    )

demand = pd.concat(frames, ignore_index=True).rename(
    columns={RAW_TIME_COL: "datetime", RAW_TARGET_COL: TARGET}
)
demand["datetime"] = pd.to_datetime(demand["datetime"])
if demand["datetime"].dt.tz is not None:
    demand["datetime"] = demand["datetime"].dt.tz_convert(TZ).dt.tz_localize(None)

demand[TARGET] = pd.to_numeric(demand[TARGET], errors="coerce")
demand = (
    demand.dropna()
    .drop_duplicates("datetime", keep="last")
    .sort_values("datetime")
    .set_index("datetime")
    .asfreq("h")
)

n_gaps = demand[TARGET].isna().sum()
demand[TARGET] = demand[TARGET].interpolate(limit=3)  # fill short gaps only (<= 3 h)
print(f"Hourly records: {len(demand):,} | gaps found: {n_gaps} | "
      f"remaining NaN: {demand[TARGET].isna().sum()}")
print(f"Range: {demand.index.min()} -> {demand.index.max()}")
demand.describe().T

[skip] demanda_mensual_SNI_2020_2025.csv: missing columns ['demanda_mw', 'fecha_hora']
Hourly records: 44,568 | gaps found: 0 | remaining NaN: 0
Range: 2020-12-01 00:00:00 -> 2025-12-31 23:00:00


,count,mean,std,min,25%,50%,75%,max
demand_mw,44568.0,3376.061313,595.810927,1914.74,2927.07,3349.34,3825.62,5202.25


In [7]:
# 2. Apparent temperature for Guayaquil, Quito and Cuenca (single Open-Meteo call)
def fetch_open_meteo(start_date: str, end_date: str) -> pd.DataFrame:
    params = {
        "latitude": ",".join(str(lat) for lat, _ in CITIES.values()),
        "longitude": ",".join(str(lon) for _, lon in CITIES.values()),
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "apparent_temperature",
        "timezone": TZ,
    }
    resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=180)
    resp.raise_for_status()
    payload = resp.json()
    payload = payload if isinstance(payload, list) else [payload]  # one entry per location, input order

    out = pd.DataFrame({"datetime": pd.to_datetime(payload[0]["hourly"]["time"])})
    for city, loc in zip(CITIES, payload):
        out[f"app_temp_{city}"] = loc["hourly"]["apparent_temperature"]
    return out.set_index("datetime")


start_date = demand.index.min().date().isoformat()
end_date = demand.index.max().date().isoformat()

weather = None
if WEATHER_CACHE.exists():
    weather = pd.read_csv(WEATHER_CACHE, index_col=0, parse_dates=True)
    weather.index.name = "datetime"
    covers = (weather.index.min().date().isoformat() <= start_date
              and weather.index.max().date().isoformat() >= end_date
              and set(f"app_temp_{c}" for c in CITIES) <= set(weather.columns))
    if not covers:
        weather = None
if weather is None:
    weather = fetch_open_meteo(start_date, end_date)
    weather.to_csv(WEATHER_CACHE)

weather = weather.interpolate(limit=3)
print(f"Weather records: {len(weather):,} | NaN: {int(weather.isna().sum().sum())}")
weather.describe().T

Weather records: 44,568 | NaN: 0


,count,mean,std,min,25%,50%,75%,max
app_temp_guayaquil,44568.0,28.533708,3.734283,20.0,25.6,28.6,31.1,40.1
app_temp_quito,44568.0,10.931751,3.412301,-0.3,8.6,10.4,13.2,22.6
app_temp_cuenca,44568.0,12.949832,4.081458,-1.4,10.1,12.4,15.8,26.7


In [8]:
# 3. Inner join demand + weather on datetime
df = demand.join(weather, how="inner")
print(f"Merged dataset: {df.shape}")
df.head()

Merged dataset: (44568, 4)


,demand_mw,app_temp_guayaquil,app_temp_quito,app_temp_cuenca
datetime,,,,
2020-12-01 00:00:00,2616.94,24.8,9.9,11.7
2020-12-01 01:00:00,2469.73,24.7,9.9,12.7
2020-12-01 02:00:00,2426.09,24.6,8.8,12.3
2020-12-01 03:00:00,2415.04,24.9,8.5,12.5
2020-12-01 04:00:00,2403.44,25.0,7.8,12.6


In [9]:
# 4a. Demand lags (lag_1h ... lag_95h) — timestamp-aligned shift, robust to gaps
#     Each day-ahead model selects the 72 lags that are already known at its forecast origin.
lags = pd.DataFrame(
    {f"lag_{k}h": df[TARGET].shift(k, freq="h").reindex(df.index) for k in LAGS},
    index=df.index,
)
df = pd.concat([df, lags], axis=1)

In [10]:
# 4b. Cyclical time encoding
hour = df.index.hour
dow = df.index.dayofweek  # 0 = Monday ... 6 = Sunday

df["sin_hour"] = np.sin(2 * np.pi * hour / 24)
df["cos_hour"] = np.cos(2 * np.pi * hour / 24)
df["sin_dow"] = np.sin(2 * np.pi * dow / 7)
df["cos_dow"] = np.cos(2 * np.pi * dow / 7)

In [11]:
# 4c. Day type: 0 = normal working day, 1 = normal weekend, 2 = official Ecuador national holiday
holiday_table = ecuador_holiday_dates(range(df.index.year.min(), df.index.year.max() + 1), EXTRA_HOLIDAYS)
print(holiday_table[holiday_table["date"].dt.year.between(2021, 2025)].to_string(index=False))

day = df.index.normalize()
df["is_holiday"] = day.isin(holiday_table["date"]).astype(int)
df["is_weekend"] = (dow >= 5).astype(int)
df["day_type"] = np.select(
    [df["is_holiday"] == 1, df["is_weekend"] == 1],
    [2, 1],
    default=0,
).astype(int)

# Mean hourly load per day type: holidays must show their own drop, distinct from a standard Sunday
df.groupby("day_type")[TARGET].agg(["mean", "std", "count"]).rename(
    index={0: "0 - Working day", 1: "1 - Weekend", 2: "2 - National holiday"}
)

      date                                         holiday
2021-02-15                                        Carnival
2021-02-16                                        Carnival
2021-04-02                                     Good Friday
2021-04-30                            Labor Day (observed)
2021-05-24                         The Battle of Pichincha
2021-08-09 Declaration of Independence of Quito (observed)
2021-10-08            Independence of Guayaquil (observed)
2021-11-01                       All Souls' Day (observed)
2021-11-03                          Independence of Cuenca
2021-12-24                        Christmas Day (observed)
2021-12-31                       New Year's Day (observed)
2022-01-01                                  New Year's Day
2022-02-28                                        Carnival
2022-03-01                                        Carnival
2022-04-15                                     Good Friday
2022-05-02                            Labor Day (observe

,mean,std,count
day_type,,,
0 - Working day,3512.479150,582.026840,30528
1 - Weekend,3093.722649,513.313298,12696
2 - National holiday,2944.519635,475.928675,1344


In [12]:
# 5. Clean and export
df_final = df.dropna().copy()
df_final.index.name = "datetime"

print(f"Final dataset: {df_final.shape} | {df_final.index.min()} -> {df_final.index.max()}")
print(f"Features: {df_final.shape[1] - 1} (target: {TARGET})")

df_final.reset_index().to_csv(OUTPUT_PATH, index=False)
print(f"Saved -> {OUTPUT_PATH}")

Final dataset: (44473, 106) | 2020-12-04 23:00:00 -> 2025-12-31 23:00:00
Features: 105 (target: demand_mw)
Saved -> /content/drive/MyDrive/Project_IA/data/dataset_stlf_ecuador.csv
